## <b> valor intrínseco y valor extrínseco.  </b>

Para cada contrato calcula:

$ I_{\text{call}} = \max(S - K, 0), \qquad I_{\text{put}} = \max(K - S, 0) $ 

Después calcula cuánto del precio corresponde a valor por encima del ejercicio inmediato:

$ E_{\text{mid}} = \text{mid} - I $ 

**Ejemplo:** con SPY en 500, una call de strike 490 y mid de 13 tiene **10 de valor intrínseco y 3 de valor extrínseco**.

La pregunta del ejercicio sería: **¿hay cotizaciones aparentemente por debajo del valor de ejercer inmediatamente y persisten cuando usamos el ask?** Una opción americana, bajo precios contemporáneos y sin fricciones, debe valer al menos su intrínseco porque permite ejercer ahora.

Tu tarea sería:

1. Clasificar los contratos como ITM, ATM u OTM.
2. Calcular intrínseco y extrínseco usando mid, bid y ask.
3. Explicar tres casos: uno OTM, uno ITM con extrínseco positivo y, si existe, uno con extrínseco negativo.

Esto ayuda a entender qué estás pagando al comprar una opción y qué valor sacrificas al ejercerla. En nuestros datos, un valor negativo sería una bandera para investigar: el cierre de SPY y las cotizaciones de opciones podrían corresponder a horas diferentes.


In [1]:
from pathlib import Path
from collections import defaultdict
from datetime import date
import gzip
import json
import pandas as pd

# Lista editable para el ejercicio.
exercise_dates = ['2020-01-02', '2022-10-03', '2025-04-03']

cwd = Path.cwd().resolve()
repo_root = next((parent for parent in (cwd, *cwd.parents) if (parent / 'quant-research-notes').is_dir()), None)
if repo_root is None:
    raise FileNotFoundError('No se encontró la raíz del repositorio: se esperaba una carpeta quant-research-notes.')

hmm_root = repo_root / 'quant-research-notes' / 'HMM_regime_options'
raw_dir = hmm_root / 'data' / 'raw'
price_path = hmm_root / 'data' / 'derived' / 'spy_daily_adjusted_2020_2026.csv'
if not raw_dir.is_dir():
    raise FileNotFoundError(f'No existe el directorio de cadenas originales: {raw_dir}')
if not price_path.is_file():
    raise FileNotFoundError(f'No existe la tabla local de cierres SPY: {price_path}')

prices = pd.read_csv(price_path, parse_dates=['date']).set_index('date')
if 'close' not in prices.columns:
    raise KeyError(f'Falta el campo close en {price_path}')

def load_chain(chain_date):
    candidates = [raw_dir / f'spy_historical_options_{chain_date}.json.gz', raw_dir / f'spy_historical_options_{chain_date}.json']
    chain_path = next((path for path in candidates if path.is_file()), None)
    if chain_path is None:
        raise FileNotFoundError(f'Falta la cadena original de SPY para {chain_date}; se buscó .json.gz y .json en {raw_dir}')
    opener = gzip.open if chain_path.suffix == '.gz' else open
    with opener(chain_path, 'rt', encoding='utf-8') as handle:
        payload = json.load(handle)
    records = payload.get('data')
    if not isinstance(records, list):
        raise KeyError(f'La cadena {chain_path.name} no contiene una lista data de contratos')
    return chain_path, records

work_frames = []
required_fields = {'contractID', 'type', 'expiration', 'strike', 'bid', 'ask'}
for chain_date in exercise_dates:
    observation_date = pd.Timestamp(chain_date)
    if observation_date not in prices.index:
        raise KeyError(f'No hay cierre SPY para {chain_date} en {price_path.name}')
    spy_close = prices.loc[observation_date, 'close']
    chain_path, raw_records = load_chain(chain_date)

    grouped = defaultdict(list)
    for record in raw_records:
        missing = required_fields.difference(record)
        if missing:
            raise KeyError(f'Faltan campos {sorted(missing)} en la cadena {chain_path.name}; contractID={record.get("contractID")}')
        grouped[str(record['contractID'])].append(record)
    deduplicated = []
    for contract_id, records in grouped.items():
        fingerprints = {json.dumps(record, sort_keys=True, separators=(',', ':'), ensure_ascii=False) for record in records}
        if len(fingerprints) > 1:
            raise RuntimeError(f'Conflicto: contractID {contract_id} tiene registros distintos en {chain_path.name}; el ejercicio se detuvo.')
        deduplicated.append(records[0])

    expiration_sides = defaultdict(set)
    for record in deduplicated:
        try:
            expiration = date.fromisoformat(str(record['expiration']))
            dte = (expiration - observation_date.date()).days
        except ValueError as exc:
            raise ValueError(f'Vencimiento inválido en {chain_path.name}: {record["expiration"]}') from exc
        if 27 <= dte <= 33 and record['type'] in {'call', 'put'}:
            expiration_sides[str(record['expiration'])].add(record['type'])
    common_expirations = [expiration for expiration, sides in expiration_sides.items() if sides == {'call', 'put'}]
    if not common_expirations:
        raise ValueError(f'No hay vencimiento común call/put entre 27 y 33 DTE para {chain_date} en {chain_path.name}')
    selected_expiration = min(
        common_expirations,
        key=lambda expiration: (abs((date.fromisoformat(expiration) - observation_date.date()).days - 30), expiration),
    )
    selected_dte = (date.fromisoformat(selected_expiration) - observation_date.date()).days
    rows = []
    for record in deduplicated:
        if record['type'] not in {'call', 'put'} or str(record['expiration']) != selected_expiration:
            continue
        try:
            strike = float(record['strike'])
        except (TypeError, ValueError) as exc:
            raise ValueError(f'Strike inválido para {record["contractID"]} en {chain_path.name}') from exc
        if abs(strike / float(spy_close) - 1.0) > 0.05:
            continue
        bid_number = pd.to_numeric(record['bid'], errors='coerce')
        ask_number = pd.to_numeric(record['ask'], errors='coerce')
        rows.append({
            'date': chain_date, 'type': record['type'], 'expiration': selected_expiration, 'DTE': selected_dte,
            'strike': strike, 'SPY_close': float(spy_close), 'bid': record['bid'], 'ask': record['ask'],
            'mid': (bid_number + ask_number) / 2, 'contractID': record['contractID'],
        })
    work_frames.append(pd.DataFrame(rows))

options_work = pd.concat(work_frames, ignore_index=True)
options_work = options_work[['date', 'type', 'expiration', 'DTE', 'strike', 'SPY_close', 'bid', 'ask', 'mid', 'contractID']]
options_work = options_work.sort_values(['date', 'type', 'strike', 'contractID']).reset_index(drop=True)
display(options_work)
print('Filas por fecha:')
display(options_work.groupby('date').size().rename('rows').reset_index())

,date,type,expiration,DTE,strike,SPY_close,bid,ask,mid,contractID
0,2020-01-02,call,2020-01-31,29,309.0,324.87,17.19,17.30,17.245,SPY200131C00309000
1,2020-01-02,call,2020-01-31,29,309.5,324.87,16.73,16.87,16.800,SPY200131C00309500
2,2020-01-02,call,2020-01-31,29,310.0,324.87,16.27,16.39,16.330,SPY200131C00310000
3,2020-01-02,call,2020-01-31,29,310.5,324.87,15.81,15.94,15.875,SPY200131C00310500
4,2020-01-02,call,2020-01-31,29,311.0,324.87,15.35,15.48,15.415,SPY200131C00311000
...,...,...,...,...,...,...,...,...,...,...
261,2025-04-03,put,2025-05-02,29,560.0,536.70,24.81,27.17,25.990,SPY250502P00560000
262,2025-04-03,put,2025-05-02,29,561.0,536.70,25.49,28.79,27.140,SPY250502P00561000
263,2025-04-03,put,2025-05-02,29,562.0,536.70,26.18,29.42,27.800,SPY250502P00562000
264,2025-04-03,put,2025-05-02,29,562.5,536.70,26.53,29.74,28.135,SPY250502P00562500


Filas por fecha:


,date,rows
0,2020-01-02,112
1,2022-10-03,70
2,2025-04-03,84
